In [1]:
# this notebook connects with a remote model and also 
# defines custom tools

In [1]:
from smolagents import CodeAgent, tool
from smolagents.models import InferenceClientModel
import os
import datetime
import requests
import pytz
import yaml
from tools.final_answer import FinalAnswerTool

ModuleNotFoundError: No module named 'smolagents'

In [12]:
HF_TOKEN = os.getenv("HF_TOKEN")

In [13]:
model = InferenceClientModel(
max_tokens=2096,
temperature=0.5,
token=HF_TOKEN,
model_id='Qwen/Qwen2.5-Coder-32B-Instruct',# it is possible that this model may be overloaded
custom_role_conversions=None,
)


In [14]:
final_answer = FinalAnswerTool()

In [15]:
@tool
def repo_owner_name()-> str: #it's import to specify the return type
    """A simple tool that returns the full name of owner of this repo
    Args:
        arg1: the first argument
        arg2: the second argument
    """
    return "Mohammad Raeez"

In [16]:
@tool
def get_current_time_in_timezone(timezone: str) -> str:
    """A tool that fetches the current local time in a specified timezone.
    Args:
        timezone: A string representing a valid timezone (e.g., 'America/New_York').
    """
    try:
        # Create timezone object
        tz = pytz.timezone(timezone)
        # Get current time in that timezone
        local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        return f"The current local time in {timezone} is: {local_time}"
    except Exception as e:
        return f"Error fetching time for timezone '{timezone}': {str(e)}"

In [17]:
with open("prompts.yaml", 'r') as stream:
    prompt_templates = yaml.safe_load(stream)

In [18]:
agent = CodeAgent(
    model=model,
    tools=[final_answer, repo_owner_name,get_current_time_in_timezone], ## add your tools here (don't remove final answer)
    max_steps=6,
    verbosity_level=2,
    planning_interval=None,
    name=None,
    description=None,
    # prompt_templates=prompt_templates
)

In [19]:
r = agent.run("What is the name of owner")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the name of owner                                                                                       │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: To find the name of the owner, I will use the `repo_owner_name` tool, which is designed to return the full
name of the owner of the repository.                                                                               
                                                                                                                   
<code>                                                                                                             
owner_name = repo_owner_name()                                                                                     
final_answer(owner_name)                                                                                           
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  owner_name = repo_owner_name()                                                                                   
  final_answer(owner_name)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Mohammad Raeezi

[Step 1: Duration 3.60 seconds| Input tokens: 2,110 | Output tokens: 54]

In [20]:
r

'Mohammad Raeezi'

In [21]:
t = agent.run("What is the time now in London")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the time now in London                                                                                  │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: To find the current time in London, I will use the `get_current_time_in_timezone` tool with the           
appropriate timezone parameter.                                                                                    
                                                                                                                   
<code>                                                                                                             
london_time = get_current_time_in_timezone(timezone="Europe/London")                                               
final_answer(london_time)                                                                                          
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  london_time = get_current_time_in_timezone(timezone="Europe/London")                                             
  final_answer(london_time)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The current local time in Europe/London is: 2026-02-23 16:48:18i

[Step 1: Duration 3.29 seconds| Input tokens: 2,111 | Output tokens: 56]

In [22]:
t

'The current local time in Europe/London is: 2026-02-23 16:48:18i'